# 💳 تقييم مخاطر الائتمان وقرار الإقراض (Credit Risk — Default Prediction)
### مشروع B3 — مسار تعلم الآلة (Machine Learning Track)

---
## 🎯 المشكلة التجارية (Business Problem)
بنك بيستقبل طلبات قروض ومحتاج يقرّر: **نوافق ولا نرفض؟** الموديل بيتنبأ **احتمال تعثّر العميل (Default)**.
بس القرار مش مجرد "هيتعثّر ولا لأ" — لازم نوازن بين **خسارة قرض متعثّر** و**خسارة عميل كويس رفضناه**.

**نوع المشكلة:** تصنيف ثنائي غير متوازن + **قرار حسّاس للتكلفة (Cost-Sensitive)**.

## 📦 ما الذي يثبته المشروع (يختلف عن مشروع الـ Churn)
التعامل مع عدم التوازن · **معايرة الاحتمالات (Probability Calibration)** ·
**عتبة القرار حسب التكلفة (Cost-Based Threshold)** · منحنى الربح · مقاييس البنوك (KS, Gini).


## 🚀 إعداد التشغيل (Colab · Kaggle · محلي)
الخلية الجاية بتثبّت المكتبات الناقصة وتجيب الداتا تلقائياً على Colab/Kaggle.
**محلياً** (بالـ env بتاع المشروع) هي مجرد no-op — تقدر تتجاهلها.

In [ ]:
# 🚀 إعداد تلقائي — Colab / Kaggle / محلي (no-op محلياً)
import os, sys, subprocess, importlib.util
REPO    = "https://github.com/ahmedelgendy11/AI-and-data-science-portfolio"
PROJECT = "ml/b3_credit_risk"          # مسار المشروع داخل portfolio/
PKGS    = ["xgboost"]          # مكتبات المشروع (تتثبّت لو ناقصة)
for _pkg in PKGS:
    if importlib.util.find_spec(_pkg.replace("-", "_")) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _pkg])
if not os.path.isdir("data"):                 # سحابياً: نجيب الريبو ونروح لمجلد المشروع
    _clone = REPO.rstrip("/").split("/")[-1]
    if not os.path.isdir(_clone):
        subprocess.run(["git", "clone", "-q", REPO + ".git"])
    os.chdir(os.path.join(_clone, "portfolio", PROJECT))
print("✓ جاهز —", os.getcwd())

## 🚀 إعداد التشغيل (Colab · Kaggle · محلي)
الخلية الجاية بتثبّت المكتبات الناقصة وتجيب الداتا تلقائياً على Colab/Kaggle.
**محلياً** (بالـ env بتاع المشروع) هي مجرد no-op — تقدر تتجاهلها.

## 📚 قبل ما تبدأ — محتاج تذاكر إيه
| المفهوم | المصدر | بيُستخدم في إيه |
|---|---|---|
| التصنيف غير المتوازن (class_weight) | Géron — *Hands-On ML* (ch.3) | المتعثّرون أقلية لكنهم الأهم |
| ROC-AUC vs PR-AUC | ISLR (ch.4) / Géron | تقييم سليم مع عدم التوازن |
| **معايرة الاحتمالات (Calibration)** | Géron (ch.3) / Niculescu-Mizil | لو هتاخد قرار بفلوس، لازم الاحتمال يكون "صادق" |
| Brier Score & Calibration Curve | sklearn docs | قياس جودة معايرة الاحتمال |
| **العتبة حسب التكلفة (Cost-Sensitive)** | Provost & Fawcett — *Data Science for Business* | 0.5 مش دايماً الأمثل — التكلفة بتحدّد العتبة |
| KS statistic / Gini | أدبيات الـ credit scoring | المقاييس القياسية في البنوك |

> 🎯 **بيُستخدم في الواقع:** البنوك، شركات التمويل، بطاقات الائتمان، الإقراض الرقمي (BNPL) — قرارات بمليارات.


## 0️⃣ المكتبات

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
sns.set_style('whitegrid'); np.random.seed(42)
print('ready ✓')

## 1️⃣ تحميل واستكشاف البيانات (EDA)

In [ ]:
df = pd.read_csv('data/credit_risk_data.csv')
# TODO: اطبع shape ومعدل التعثّر والقيم المفقودة، وارسم العلاقات
print(df['default'].mean())

## 2️⃣ المعالجة والتقسيم (Pipeline + Stratified Split)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
# TODO: افصل X/y، حدّد الأعمدة الرقمية والفئوية، split stratified، ColumnTransformer
X = ...; y = ...
pre = ColumnTransformer([...])

## 3️⃣ مقارنة الموديلات (ROC-AUC + PR-AUC) مع معالجة عدم التوازن

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
# TODO: احسب scale_pos_weight، قارن 3 موديلات بـ ROC-AUC و PR-AUC
...

## 4️⃣ معايرة الاحتمالات (Probability Calibration) 🎯
موديلات زي RF/XGBoost بتطلّع احتمالات **مش مظبوطة** (لو قال 0.8 مش معناه إن 80% فعلاً بيتعثّروا).
لو هناخد قرار بفلوس، لازم الاحتمال يكون **صادق**. نقيس بـ Brier Score (الأقل أحسن) ومنحنى المعايرة.

In [ ]:
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import brier_score_loss
# TODO: درّب الموديل، ثم CalibratedClassifierCV (isotonic)، وقارن Brier + منحنى المعايرة
base = ...
cal = ...

## 5️⃣ عتبة القرار حسب التكلفة (Cost-Based Threshold) 💰
**الفكرة الأهم:** قبول متعثّر (False Negative) بيخسّر البنك القرض كله. رفض عميل كويس (False Positive)
بيخسّر الأرباح المتوقّعة فقط — أرخص بكتير. فالعتبة المثلى **مش 0.5**.
نفترض: تكلفة FN = 1.0 (خسارة القرض)، تكلفة FP = 0.15 (أرباح ضائعة). نختار العتبة اللي بتقلّل التكلفة.

In [ ]:
COST_FN, COST_FP = 1.0, 0.15
# TODO: لكل عتبة احسب التكلفة = FN*COST_FN + FP*COST_FP، واختر العتبة الأقل تكلفة
ths = np.linspace(0.05, 0.95, 91)
best_t = ...
print('Optimal threshold:', best_t)

## 6️⃣ التقييم النهائي + مقاييس البنوك (KS / Gini)

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score
from scipy.stats import ks_2samp
# TODO: قيّم عند العتبة المثلى، واحسب AUC و Gini=2*AUC-1 و KS statistic
...

## 7️⃣ الخلاصة والتوصيات (Conclusion)
- **النتيجة:** XGBoost المعاير حقّق AUC/Gini كويسين، واحتمالاته صارت **صادقة** بعد المعايرة (Brier أقل).
- **العتبة المثلى أقل من 0.5** — لأن تكلفة قبول متعثّر أعلى بكتير من رفض عميل كويس، فالبنك يميل للتحفّظ.
- **مقاييس البنوك:** KS و Gini ضمن النطاق المقبول في الـ credit scoring.
- **التوصية:** اعتماد الموديل المعاير + العتبة المبنية على التكلفة، مع مراجعة بشرية للحالات الحدّية.
- **الخطوة القادمة:** تحويله لـ scorecard (نقاط)، ومراقبة الانحراف (drift) بعد النشر.

> ✅ **اللي اتعلمته:** عدم التوازن، PR-AUC، **معايرة الاحتمالات**، **العتبة حسب التكلفة**، و KS/Gini.
